# 06 — Innovations Kalman et NLL

Si le RC est bon, e_k = y_k - C xhat^-_k ressemble à du bruit blanc.
La PEM minimise la **NLL** de ces innovations (`filter_r1c1`).
Figure I3 du catalogue.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import tempfile

from basic_mpc.identification.figures import plot_innovations
from basic_mpc.identification.pem import filter_r1c1
from basic_mpc.models.r1c1 import R1C1Params, simulate_r1c1

params = R1C1Params(a=0.96, g_solar=2e-5, g_heating=4e-3,
                    process_noise_std=0.03, sensor_noise_std=0.25)
n = 800
hours = np.arange(n) * 0.25
t_ext = 8 + 6 * np.sin(2 * np.pi * hours / 24)
solar = np.clip(np.sin(2 * np.pi * (hours - 6) / 24), 0, None) * 2000
heating = np.where((hours % 24 > 7) & (hours % 24 < 22), 15.0, 0.0)
_x, y = simulate_r1c1(params, t_ext, solar, heating, x0=18.0, seed=3)
u = np.column_stack([t_ext, solar, heating])
bon = filter_r1c1(y, u, params)
mauvais = filter_r1c1(y, u, R1C1Params(a=0.80, g_solar=1e-8, g_heating=1e-8,
                                      process_noise_std=0.03, sensor_noise_std=0.25))
print("loglik modèle vrai", bon.loglik, "n_obs", bon.n_obs)
print("loglik modèle faux", mauvais.loglik)

In [ ]:
tmp = Path(tempfile.mkdtemp()) / "i3.png"
plot_innovations(bon.innov, tmp)
from IPython.display import Image
Image(filename=str(tmp))